In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="google/medgemma-1.5-4b-it",
)


C:\Users\Professor\anaconda3\envs\dl-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
# Instale antes:
# pip install -U transformers accelerate torch pillow requests

from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import requests
import torch
from io import BytesIO

model_id = "google/medgemma-1.5-4b-it"

# Carrega o modelo
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Carrega o processador
processor = AutoProcessor.from_pretrained(model_id)

# Imagem de exemplo
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"

response = requests.get(
    image_url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
)

response.raise_for_status()

image = Image.open(BytesIO(response.content)).convert("RGB")

# Prompt
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": "Describe this X-ray"
            },
        ],
    }
]

# Processa entrada
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

# Geração
with torch.inference_mode():
    generation = model.generate(
        **inputs,
        max_new_tokens=2000,
        do_sample=False,
    )

generation = generation[0][input_len:]

# Decodifica
decoded = processor.decode(
    generation,
    skip_special_tokens=True,
)

print("\nResposta do MedGemma:\n")
print(decoded)
